In [2]:
import sys
import os
import urllib3
from configparser import ConfigParser

# Add your local ThreatConnect SDK to path
sys.path.append("Z:/HTOC/Data_Analytics/threatconnect")
from ThreatConnect import ThreatConnect
from RequestObject import RequestObject
from Owners import Owners

# Add your project repo to path
project_root = r"C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull"
if project_root not in sys.path:
    sys.path.append(project_root)

from utils.config_loader import load_config

# Load API config
config_path = os.path.join(project_root, "utils", "config.json")
try:
    api_secret_key, api_access_id, api_base_url, api_default_org = load_config(config_path)
    print(f"Loaded config from: {config_path}")
    print(f"Base URL: {api_base_url}")
    print(f"Access ID: {api_access_id}")
    print(f"Default Org: {api_default_org}")
except Exception as e:
    print(f"[ERROR] Failed to load configuration: {e}")
    sys.exit(1)

# Disable SSL verification warnings (use cautiously)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
verify_ssl = False

# Initialize ThreatConnect session
try:
    tc = ThreatConnect(api_access_id, api_secret_key, api_default_org, api_base_url)
    print("ThreatConnect initialized.")
except Exception as e:
    print(f"[ERROR] Failed to initialize ThreatConnect: {e}")
    sys.exit(1)

# Define the owner (organization scope)
owner = 'HTOC Org'

# Create a request object to fetch indicators (or other data)
try:
    ro = RequestObject()
    ro.set_http_method('GET')
    ro.set_owner(owner)
    ro.set_owner_allowed(True)
    # ro.set_resource_pagination(True)  # Uncomment if needed
    print("RequestObject successfully created.")
except Exception as e:
    print(f"[ERROR] Failed to initialize RequestObject: {e}")
    sys.exit(1)



Loaded config from: C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull\utils\config.json
Base URL: https://hvs.threatconnect.com/api
Access ID: 09783848890162390382
Default Org: HTOC Org
ThreatConnect initialized.
RequestObject successfully created.


In [3]:
import pandas as pd
from datetime import datetime, timedelta
import pytz
import urllib.parse

# Setup
cutoff = pd.Timestamp.utcnow()
start_date = (datetime.now(pytz.UTC) - timedelta(days=3)).date()
start = f"{start_date}T00:00:00Z"
type_names = ["Address", "EmailAddress", "File", "Host", "URL", "ASN", "CIDR",
              "Email Subject", "Hashtag", "Mutex", "Registry Key", "Stripped URL", "User Agent"]
type_name_condition = ", ".join([f'"{t}"' for t in type_names])
list_of_owners = ['HTOC Org']
final_results = []

# Query indicators
for owner in list_of_owners:
    print(f"Querying owner: {owner}")
    try:
        tql_raw = (
            f'ownerName EQ "{owner}" AND '
            f'typeName IN ({type_name_condition}) AND '
            f'lastObserved >= "{start}"'
        )
        tql_encoded = urllib.parse.quote(tql_raw)
        ro.set_http_method('GET')
        ro.set_request_uri(f'/v3/indicators?tql={tql_encoded}&fields=tags,observations&resultStart=0&resultLimit=10000')
        response = tc.api_request(ro)

        if response.headers.get('content-type') == 'application/json':
            results = response.json()
            final_results.append(results)
    except Exception as e:
        print(f"Failed to query indicators for {owner}: {e}")

# Normalize results
normalized_data = []
for result in final_results:
    for item in result.get('data', []):
        if 'summary' in item:
            normalized_data.append(item)

observed_src = pd.json_normalize(normalized_data)
observed_src['indicator'] = observed_src['summary'].str.split().str[0].str.strip()
observed_src.drop_duplicates(subset='indicator', inplace=True)
observed_src['lastObserved'] = pd.to_datetime(observed_src['lastObserved'], utc=True)
observed_src = observed_src[observed_src['lastObserved'] >= pd.to_datetime(start)]


Querying owner: HTOC Org


In [4]:
import os
import pandas as pd
from datetime import datetime, timedelta

# Base file path with placeholder for date
base_path = r"Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d{date}.csv"
#base_path = r"C:\Users\jaskew\Documents\project_repository\data\raw\ObservationDataFiles\htoc_opdiv_obs_d{date}.csv"
date_format = "%Y%m%d"

def get_file_paths(base_path, days=3):
    """ Generate file paths for the last `days` days using list comprehension. """
    today = datetime.utcnow()
    dates_to_pull = [(today - timedelta(days=i)).strftime(date_format) for i in range(days)]
    
    # Construct file paths
    file_paths = [base_path.format(date=dt) for dt in dates_to_pull]
    
    # Filter for existing files
    existing_files = [file_path for file_path in file_paths if os.path.exists(file_path)]
    
    if not existing_files:
        print("No files found for the specified date range.")
    else:
        print(f"Files to be loaded: {existing_files}")
    
    return existing_files

def load_observed_data(file_paths):
    """ Load and concatenate observed data from multiple files. """
    data_frames = []

    for file_path in file_paths:
        try:
            df = pd.read_csv(file_path)
            data_frames.append(df)
        except Exception as e:
            print(f"Error reading file {file_path}: {e}")
    
    # Concatenate data
    if data_frames:
        observed_data_df = pd.concat(data_frames, ignore_index=True)
        print(f"Loaded data from {len(data_frames)} files.")
    else:
        observed_data_df = pd.DataFrame()

    return observed_data_df

# Example Usage:
# Fetch file paths for the last 3 days
file_paths = get_file_paths(base_path, days=3)

# Load observed data
observed_data_df = load_observed_data(file_paths)



Files to be loaded: ['Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250618.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250617.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250616.csv']
Loaded data from 3 files.


In [41]:
import warnings

# Extract API-tagged indicators
all_filtered = []
cutoff = pd.Timestamp.utcnow()

for _, row in observed_src.iterrows():
    tags_data = row.get('tags.data')
    if isinstance(tags_data, list):
        tags_df = pd.json_normalize(tags_data)
        api_tags = tags_df[tags_df['name'].str.contains('API', case=False, na=False)]
        if not api_tags.empty:
            all_tags_list = tags_df['name'].astype(str).tolist()
            api_tags['name'] = api_tags['name'].astype(str)
            for col in ['summary', 'observations', 'description', 'type', 'dateAdded', 'lastModified', 'lastObserved', 'webLink', 'id']:
                api_tags[col] = row.get(col)
            api_tags['all_tags'] = [all_tags_list] * len(api_tags)
            all_filtered.append(api_tags)
            warnings.simplefilter(action='ignore', category=pd.errors.SettingWithCopyWarning)
filtered_tags = pd.concat(all_filtered, ignore_index=True)
filtered_tags['lastObserved'] = pd.to_datetime(filtered_tags['lastObserved'], errors='coerce')
filtered_tags['dateAdded'] = pd.to_datetime(filtered_tags['dateAdded'], errors='coerce')

# Prepare for matching
filtered_tags['OpDiv'] = filtered_tags['name'].str.replace(' Splunk API', '', regex=False)
obs_subset = observed_data_df[['indicator', 'OpDiv', 'obs_date']].drop_duplicates()
recent_tags = pd.merge(filtered_tags, obs_subset, left_on=['summary', 'OpDiv'], right_on=['indicator', 'OpDiv'], how='inner')

# Filter to recent
cutoff_naive = cutoff.tz_convert(None)
recent_tags = recent_tags[
    (recent_tags['lastObserved'] >= cutoff - timedelta(hours=24)) &
    (pd.to_datetime(recent_tags['obs_date'], errors='coerce') >= cutoff_naive - timedelta(days=1))
].copy()

# Partner processing
recent_tags['partner'] = recent_tags['name'].str.replace(' Splunk API', '', regex=False)
partner_groups = (
    recent_tags.groupby('summary')['partner']
    .agg(['nunique', lambda x: ', '.join(sorted(set(x)))])
    .reset_index()
    .rename(columns={'nunique': 'partner_count', '<lambda_0>': 'partners'})
)
recent_tags = recent_tags.merge(partner_groups, on='summary', how='left')
recent_tags.drop_duplicates(subset='summary', inplace=True)


In [42]:
# Enrich only final filtered indicators
indicator_values = recent_tags['summary'].dropna().unique().tolist()
enriched_results = []

print(f"Enriching {len(indicator_values)} indicators with DomainTools and VirusTotalV3...")

for value in indicator_values:
    try:
        # Use the indicator *value*, not the ID
        encoded_value = urllib.parse.quote(value)
        enrich_url = f'/v3/indicators/{encoded_value}/enrich?type=Shodan&type=VirusTotalV3'
        ro.set_http_method('POST')
        ro.set_request_uri(enrich_url)
        ro.set_body({})
        enrich_response = tc.api_request(ro)

        if enrich_response.status_code == 200:
            enrich_data = enrich_response.json()
            enrich_data['summary'] = value  # Save the value as key
            enriched_results.append(enrich_data)
    except Exception as e:
        continue

if enriched_results:
    df_enriched = pd.json_normalize(enriched_results)
    df_enriched = pd.json_normalize(enriched_results)
    recent_tags = recent_tags.merge(df_enriched, on='summary', how='left')
    # Unnest 'data.enrichment.data' (list of dicts) into separate columns
if 'data.enrichment.data' in recent_tags.columns:
    enrichment_df = pd.json_normalize(
        recent_tags['data.enrichment.data'].dropna().explode()
    )
    enrichment_df.index = recent_tags['data.enrichment.data'].dropna().explode().index
    enrichment_cols = [col for col in enrichment_df.columns if col not in recent_tags.columns]
    # Join enrichment columns back to recent_tags
    recent_tags = recent_tags.join(enrichment_df[enrichment_cols], how='left')

    # Keep only records with vtMaliciousCount > 10
    recent_tags = recent_tags[recent_tags['vtMaliciousCount'] > 10]

    print(f"Successfully enriched and merged {len(df_enriched)} indicators.")
else:
    print("No enrichment data retrieved.")


Enriching 413 indicators with DomainTools and VirusTotalV3...


Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The Host uploads-ssl.webflow.com cannot be enriched with Shodan because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The Host irp.cdn-website.com cannot be enriched with Shodan because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The Host ctrk.klclick2.com cannot be enriched with Shodan because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The Host omnicell.com cannot be enriched with Shodan because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The Host rqipartners.com cannot be enriched with Shodan because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API R

Successfully enriched and merged 390 indicators.


In [60]:
import numpy as np

# Unnest the 'data.enrichment.data' column into separate columns for each enrichment type

def extract_enrichment(row):
    """Extracts enrichment fields from the list of dicts in 'data.enrichment.data'."""
    enrichments = row.get('data.enrichment.data')
    result = {}
    if isinstance(enrichments, list):
        for enrich in enrichments:
            enrich_type = enrich.get('type')
            if enrich_type == 'Shodan':
                # Flatten Shodan fields
                for key in ['hostNames', 'domains', 'tags', 'country', 'city', 'isp', 'asn', 'org', 'openPorts']:
                    result[f'shodan_{key}'] = enrich.get(key, np.nan)
            elif enrich_type == 'VirusTotal':
                result['vtMaliciousCount'] = enrich.get('vtMaliciousCount', np.nan)
    return pd.Series(result)

# Apply extraction to recent_tags
enrichment_expanded = recent_tags.apply(extract_enrichment, axis=1)
recent_tags = pd.concat([recent_tags, enrichment_expanded], axis=1)
recent_tags = recent_tags[
    [
        'indicator',
        'vtMaliciousCount',
        'obs_date',
        'shodan_asn',
        'shodan_city',
        'shodan_country',
        'data.legacyLink',
        'partners'
    ]
]

recent_tags = recent_tags.rename(columns={
    'indicator': 'Indicator',
    'vtMaliciousCount': 'Malicious Score/Count',
    'obs_date': 'Observation Date',
    'shodan_asn': 'ASN',
    'shodan_city': 'City',
    'shodan_country': 'Country',
    'data.legacyLink': 'ThreatConnect Link',
    'partners': 'Partners'
})
recent_tags = recent_tags.fillna('unknown')
recent_tags = recent_tags.drop_duplicates()
recent_tags

,Indicator,Malicious Score/Count,Observation Date,ASN,City,Country,ThreatConnect Link,Partners
7,109.70.100.2,11.0,2025-06-18,AS208323,Vienna,Austria,https://hvs.threatconnect.com/auth/indicators/...,CMS
14,78.153.140.177,13.0,2025-06-18,AS202306,London,United Kingdom,https://hvs.threatconnect.com/auth/indicators/...,"FDA, IHS, OS"
18,45.95.146.59,11.0,2025-06-18,AS49870,Lelystad,Netherlands,https://hvs.threatconnect.com/auth/indicators/...,"CMS, DHA, FDA, HRSA, IHS"
19,167.99.13.19,12.0,2025-06-18,AS14061,North Bergen,United States,https://hvs.threatconnect.com/auth/indicators/...,"DHA, FDA, HRSA, IHS, OS"
25,178.128.84.112,12.0,2025-06-18,AS14061,Singapore,Singapore,https://hvs.threatconnect.com/auth/indicators/...,"DHA, FDA, HRSA, OS"
...,...,...,...,...,...,...,...,...
363,104.167.221.114,19.0,2025-06-18,AS399045,Kansas City,United States,https://hvs.threatconnect.com/auth/indicators/...,"CMS, DHA, HRSA, IHS, NIH, OS"
365,118.193.72.187,11.0,2025-06-18,unknown,unknown,unknown,https://hvs.threatconnect.com/auth/indicators/...,"CMS, HHS, OS"
369,183.224.237.233,11.0,2025-06-18,AS9808,Shanghai,China,https://hvs.threatconnect.com/auth/indicators/...,"CMS, FDA, HRSA"
374,185.220.101.182,12.0,2025-06-18,AS60729,Berlin,Germany,https://hvs.threatconnect.com/auth/indicators/...,CMS


In [61]:
from IPython.display import display

# Get all unique partners (split by comma and flatten)
all_partners = set(
    p.strip()
    for partners in recent_tags['Partners'].dropna().unique()
    for p in partners.split(',')
)

# Build buckets: each partner gets all rows where it appears in the partners column
partner_buckets = {
    partner: recent_tags[recent_tags['Partners'].str.contains(rf'\b{partner}\b', na=False)]
    for partner in all_partners
}

# Show each partner's dataframe as a table
for partner, df in partner_buckets.items():
    print(f"Partner: {partner} ({len(df)} records)")
    display(df)

Partner: IHS (18 records)


,Indicator,Malicious Score/Count,Observation Date,ASN,City,Country,ThreatConnect Link,Partners
14,78.153.140.177,13.0,2025-06-18,AS202306,London,United Kingdom,https://hvs.threatconnect.com/auth/indicators/...,"FDA, IHS, OS"
18,45.95.146.59,11.0,2025-06-18,AS49870,Lelystad,Netherlands,https://hvs.threatconnect.com/auth/indicators/...,"CMS, DHA, FDA, HRSA, IHS"
19,167.99.13.19,12.0,2025-06-18,AS14061,North Bergen,United States,https://hvs.threatconnect.com/auth/indicators/...,"DHA, FDA, HRSA, IHS, OS"
31,91.191.209.198,13.0,2025-06-18,AS57509,Sofia,Bulgaria,https://hvs.threatconnect.com/auth/indicators/...,"CMS, DHA, FDA, HRSA, IHS, OS"
57,78.153.140.179,14.0,2025-06-18,AS202306,London,United Kingdom,https://hvs.threatconnect.com/auth/indicators/...,"FDA, IHS, OS"
71,142.93.115.5,13.0,2025-06-18,AS14061,North Bergen,United States,https://hvs.threatconnect.com/auth/indicators/...,"CMS, DHA, FDA, HRSA, IHS"
93,80.67.167.81,14.0,2025-06-18,AS2027,Paris,France,https://hvs.threatconnect.com/auth/indicators/...,"CMS, FDA, HRSA, IHS, OS"
95,185.169.4.150,20.0,2025-06-18,AS209605,Kaunas,Lithuania,https://hvs.threatconnect.com/auth/indicators/...,"CMS, DHA, FDA, HRSA, IHS, OS"
104,152.32.128.214,11.0,2025-06-18,unknown,unknown,unknown,https://hvs.threatconnect.com/auth/indicators/...,"CMS, DHA, FDA, HHS, HRSA, IHS"
192,71.6.232.25,11.0,2025-06-18,unknown,unknown,unknown,https://hvs.threatconnect.com/auth/indicators/...,"CMS, DHA, FDA, HRSA, IHS, OS"


Partner: HHS (4 records)


,Indicator,Malicious Score/Count,Observation Date,ASN,City,Country,ThreatConnect Link,Partners
35,64.227.146.243,11.0,2025-06-18,AS14061,Doddaballapura,India,https://hvs.threatconnect.com/auth/indicators/...,"CMS, HHS"
104,152.32.128.214,11.0,2025-06-18,unknown,unknown,unknown,https://hvs.threatconnect.com/auth/indicators/...,"CMS, DHA, FDA, HHS, HRSA, IHS"
116,165.22.54.16,11.0,2025-06-18,unknown,unknown,unknown,https://hvs.threatconnect.com/auth/indicators/...,"CMS, HHS"
365,118.193.72.187,11.0,2025-06-18,unknown,unknown,unknown,https://hvs.threatconnect.com/auth/indicators/...,"CMS, HHS, OS"


Partner: CDC (1 records)


...

,Indicator,Malicious Score/Count,Observation Date,ASN,City,Country,ThreatConnect Link,Partners
14,78.153.140.177,13.0,2025-06-18,AS202306,London,United Kingdom,https://hvs.threatconnect.com/auth/indicators/...,"FDA, IHS, OS"
19,167.99.13.19,12.0,2025-06-18,AS14061,North Bergen,United States,https://hvs.threatconnect.com/auth/indicators/...,"DHA, FDA, HRSA, IHS, OS"
25,178.128.84.112,12.0,2025-06-18,AS14061,Singapore,Singapore,https://hvs.threatconnect.com/auth/indicators/...,"DHA, FDA, HRSA, OS"
31,91.191.209.198,13.0,2025-06-18,AS57509,Sofia,Bulgaria,https://hvs.threatconnect.com/auth/indicators/...,"CMS, DHA, FDA, HRSA, IHS, OS"
39,104.152.52.148,11.0,2025-06-18,unknown,unknown,unknown,https://hvs.threatconnect.com/auth/indicators/...,"FDA, OS"
40,171.25.193.20,16.0,2025-06-18,AS198093,Stockholm,Sweden,https://hvs.threatconnect.com/auth/indicators/...,OS
57,78.153.140.179,14.0,2025-06-18,AS202306,London,United Kingdom,https://hvs.threatconnect.com/auth/indicators/...,"FDA, IHS, OS"
86,185.241.208.204,13.0,2025-06-18,AS210558,Warsaw,Poland,https://hvs.threatconnect.com/auth/indicators/...,OS
93,80.67.167.81,14.0,2025-06-18,AS2027,Paris,France,https://hvs.threatconnect.com/auth/indicators/...,"CMS, FDA, HRSA, IHS, OS"
95,185.169.4.150,20.0,2025-06-18,AS209605,Kaunas,Lithuania,https://hvs.threatconnect.com/auth/indicators/...,"CMS, DHA, FDA, HRSA, IHS, OS"


Partner: DHA (18 records)


,Indicator,Malicious Score/Count,Observation Date,ASN,City,Country,ThreatConnect Link,Partners
18,45.95.146.59,11.0,2025-06-18,AS49870,Lelystad,Netherlands,https://hvs.threatconnect.com/auth/indicators/...,"CMS, DHA, FDA, HRSA, IHS"
19,167.99.13.19,12.0,2025-06-18,AS14061,North Bergen,United States,https://hvs.threatconnect.com/auth/indicators/...,"DHA, FDA, HRSA, IHS, OS"
25,178.128.84.112,12.0,2025-06-18,AS14061,Singapore,Singapore,https://hvs.threatconnect.com/auth/indicators/...,"DHA, FDA, HRSA, OS"
31,91.191.209.198,13.0,2025-06-18,AS57509,Sofia,Bulgaria,https://hvs.threatconnect.com/auth/indicators/...,"CMS, DHA, FDA, HRSA, IHS, OS"
71,142.93.115.5,13.0,2025-06-18,AS14061,North Bergen,United States,https://hvs.threatconnect.com/auth/indicators/...,"CMS, DHA, FDA, HRSA, IHS"
95,185.169.4.150,20.0,2025-06-18,AS209605,Kaunas,Lithuania,https://hvs.threatconnect.com/auth/indicators/...,"CMS, DHA, FDA, HRSA, IHS, OS"
104,152.32.128.214,11.0,2025-06-18,unknown,unknown,unknown,https://hvs.threatconnect.com/auth/indicators/...,"CMS, DHA, FDA, HHS, HRSA, IHS"
113,193.163.125.127,11.0,2025-06-18,AS211298,Manchester,United Kingdom,https://hvs.threatconnect.com/auth/indicators/...,"CMS, DHA, FDA, HRSA, OS"
192,71.6.232.25,11.0,2025-06-18,unknown,unknown,unknown,https://hvs.threatconnect.com/auth/indicators/...,"CMS, DHA, FDA, HRSA, IHS, OS"
202,170.64.134.89,13.0,2025-06-18,AS14061,Sydney,Australia,https://hvs.threatconnect.com/auth/indicators/...,"DHA, FDA, HRSA, OS"


Partner: HRSA (37 records)


,Indicator,Malicious Score/Count,Observation Date,ASN,City,Country,ThreatConnect Link,Partners
18,45.95.146.59,11.0,2025-06-18,AS49870,Lelystad,Netherlands,https://hvs.threatconnect.com/auth/indicators/...,"CMS, DHA, FDA, HRSA, IHS"
19,167.99.13.19,12.0,2025-06-18,AS14061,North Bergen,United States,https://hvs.threatconnect.com/auth/indicators/...,"DHA, FDA, HRSA, IHS, OS"
25,178.128.84.112,12.0,2025-06-18,AS14061,Singapore,Singapore,https://hvs.threatconnect.com/auth/indicators/...,"DHA, FDA, HRSA, OS"
31,91.191.209.198,13.0,2025-06-18,AS57509,Sofia,Bulgaria,https://hvs.threatconnect.com/auth/indicators/...,"CMS, DHA, FDA, HRSA, IHS, OS"
71,142.93.115.5,13.0,2025-06-18,AS14061,North Bergen,United States,https://hvs.threatconnect.com/auth/indicators/...,"CMS, DHA, FDA, HRSA, IHS"
93,80.67.167.81,14.0,2025-06-18,AS2027,Paris,France,https://hvs.threatconnect.com/auth/indicators/...,"CMS, FDA, HRSA, IHS, OS"
95,185.169.4.150,20.0,2025-06-18,AS209605,Kaunas,Lithuania,https://hvs.threatconnect.com/auth/indicators/...,"CMS, DHA, FDA, HRSA, IHS, OS"
99,178.20.55.16,12.0,2025-06-18,AS29075,Tremblay-en-France,France,https://hvs.threatconnect.com/auth/indicators/...,"HRSA, OS"
104,152.32.128.214,11.0,2025-06-18,unknown,unknown,unknown,https://hvs.threatconnect.com/auth/indicators/...,"CMS, DHA, FDA, HHS, HRSA, IHS"
106,185.129.62.62,11.0,2025-06-18,AS57860,Copenhagen,Denmark,https://hvs.threatconnect.com/auth/indicators/...,"CMS, HRSA"


In [62]:
import os
import re
import time
from datetime import datetime
import pandas as pd
import win32com.client as win32

Tippers_Path = r'Z:\HTOC\HTOC Reports\Tippers'
os.makedirs(Tippers_Path, exist_ok=True)

# Add today's date to the filename
today_str = datetime.utcnow().strftime("%Y%m%d")
excel_path = os.path.join(Tippers_Path, f"tippers_by_partner_{today_str}.xlsx")

with pd.ExcelWriter(excel_path, engine='xlsxwriter') as writer:
    for partner, df in partner_buckets.items():
        # Excel sheet names can't be longer than 31 chars or contain special chars
        safe_partner = re.sub(r'[^a-zA-Z0-9_-]', '_', partner)[:31]

        # Convert all timezone-aware datetime columns to naive
        for col in df.select_dtypes(include=['datetimetz']).columns:
            df[col] = df[col].dt.tz_localize(None)

        # Write to Excel first
        df.to_excel(writer, sheet_name=safe_partner, index=False)

        # Then get the worksheet object to format
        worksheet = writer.sheets[safe_partner]
        worksheet.autofilter(0, 0, len(df), len(df.columns) - 1)
        worksheet.freeze_panes(1, 0)

        for i, col in enumerate(df.columns):
            # Set width based on max text length in the column, or column name
            max_len = max(
                df[col].astype(str).map(len).max() if not df.empty else 0,
                len(str(col))
            )
            worksheet.set_column(i, i, min(max_len + 2, 50))  # Cap width at 50

print(f"Excel file with partner tabs saved to: {excel_path}")


Excel file with partner tabs saved to: Z:\HTOC\HTOC Reports\Tippers\tippers_by_partner_20250618.xlsx
